In [34]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm
import time

In [35]:
production_catalogue_unique = (
    production_catalogue
    .drop_duplicates(
        subset="production_id",
        keep="first"
    )
    .reset_index(drop=True)
)

In [36]:
production_catalogue_unique.shape

(7292, 6)

In [37]:
def extract_production_metadata(url, session):

    response = session.get(url, timeout=30)
    response.raise_for_status()

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    metadata = {}

    for label in soup.find_all(
        "div",
        class_="xt-lable"
    ):

        key = label.get_text(strip=True)

        value = label.find_next_sibling("div")

        if value:
            value = value.get_text(
                " ",
                strip=True
            )
        else:
            value = None

        metadata[key] = value

    return {
        "opening_date": metadata.get("Opening Date"),
        "closing_date": metadata.get("Closing Date"),
        "production_type": metadata.get("Production Type")
    }

In [38]:
session = requests.Session()

metadata = []

for i, row in tqdm(
    production_catalogue_unique.iterrows(),
    total=len(production_catalogue_unique)
):

    retries = 3

    for attempt in range(retries):
        try:
            data = extract_production_metadata(
                row["url"],
                session
            )

            data["production_id"] = row["production_id"]

            metadata.append(data)
            break

        except Exception as e:
            print(
                f"Error scraping {row['production_id']} "
                f"(attempt {attempt + 1}/{retries}): {e}"
            )

            if attempt < retries - 1:
                time.sleep(5)

            else:
                metadata.append({
                    "production_id": row["production_id"],
                    "opening_date": None,
                    "closing_date": None,
                    "production_type": None
                })

    # checkpoint every 100 productions
    if i % 100 == 0:
        pd.DataFrame(metadata).to_csv(
            "../data/processed/production_metadata_checkpoint.csv",
            index=False
        )

    # polite delay
    time.sleep(0.3)

 19%|█▉        | 1376/7292 [30:58<1:44:41,  1.06s/it]

Error scraping 11187 (attempt 1/3): HTTPSConnectionPool(host='www.ibdb.com', port=443): Read timed out.


 51%|█████     | 3720/7292 [1:32:13<1:15:41,  1.27s/it]

Error scraping 477821 (attempt 1/3): 429 Client Error: Too Many Requests for url: https://www.ibdb.com/broadway-production/cyrano-de-bergerac-477821


 51%|█████     | 3729/7292 [1:32:32<1:17:44,  1.31s/it]

...

 93%|█████████▎| 6815/7292 [2:48:58<11:52,  1.49s/it]

Error scraping 494965 (attempt 1/3): HTTPSConnectionPool(host='www.ibdb.com', port=443): Read timed out. (read timeout=30)


 94%|█████████▍| 6847/7292 [2:49:48<03:09,  2.34it/s]  

Error scraping 495294 (attempt 1/3): 429 Client Error: Too Many Requests for url: https://www.ibdb.com/broadway-production/hedwig-and-the-angry-inch-495294


100%|██████████| 7292/7292 [2:59:50<00:00,  1.48s/it]


In [39]:
 pd.DataFrame(metadata).to_csv(
            "../data/processed/production_metadata_checkpoint.csv",
            index=False
        )